# Add LSST Photometric Calibration Parameters to Light Curves - Version B


- note something is wrong , especially concerning calib_local. But I also suspect missing data.
- faster than notebook 05



This notebook enriches the light-curve files produced by
`02_MergeLCsourceswithMJD.ipynb` with per-exposure photometric calibration
information retrieved from the Butler.

## Important note on the detector column

The `detector` column in the input light-curve files is **unreliable** (filled
with -1). The correct detector is therefore **not** used as a Butler key.
Instead, for each data point the Butler is queried using:
- the `visit` number (exact match),
- the `band`,
- the sky position (`src_ra`, `src_dec`) via a spatial overlap on
  `visit_detector_region.region`.

This guarantees that the returned PVI is the unique detector that covered the
source during that visit. The real detector id and name are then read back
from the image and stored in the output table.

## What is added per data point

| New column | Source | Meaning |
|---|---|---|
| `calib_mean` | `photocal.getCalibrationMean()` | Mean calibration factor over the CCD [nJy / ADU] |
| `calib_err` | `photocal.getCalibrationErr()` | Uncertainty on the mean calibration factor |
| `calib_local` | `photocal.getLocalCalibration(Point2D(x, y))` | Local calibration factor at the source pixel position |
| `zeropoint` | `2.5 * log10(photocal.getInstFluxAtZeroMagnitude())` | AB zero-point magnitude |
| `visit_from_image` | `pvi.visitInfo.id` | Visit id embedded in the image (sanity check vs `visit`) |
| `detector_id` | `pvi.getDetector().getId()` | Detector number (replaces the -1 placeholder) |
| `detector_name` | `pvi.getDetector().getName()` | Detector name (e.g. `R22_S11`) |

## Strategy

1. Load the global merged light-curve file from `data_MergeVisits_02_out/`.
2. Build a deduplicated list of unique `(band, visit, src_ra, src_dec)` groups.
   For each group call `butler.query_datasets()` with a spatial + visit filter
   to locate the unique PVI, then load it and extract calib + detector info.
3. Cache results keyed on `(band, visit)` — a source lies on exactly one
   detector per visit, so the spatial query resolves the ambiguity and the
   result is reusable for all rows sharing the same `(band, visit)`.
4. Attach calibration columns (plus `calib_local` computed per-row from
   the cached `PhotoCalib` object and the source pixel coordinates `x, y`).
5. Write outputs to `data_AddCalib_05_out/`.

## Input
- `data_MergeVisits_02_out/all_stars_lightcurves_mjd.csv` (global)
- `data_MergeVisits_02_out/per_star/*_lc_mjd.csv` (per-star)

## Output
- `data_AddCalib_05_out/all_stars_lightcurves_calib.csv` + `.parquet`
- `data_AddCalib_05_out/per_star/*_lc_calib.csv` + `.parquet`

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-29
- **Last update:** 2026-06-29
- **Last update:** 2026-06-30 : correct a bug for local calib

## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
from lsst.geom import Point2D

from libExtractLightcurves import safe_name, find_col, dataset_type_exists

## 2. Logging setup

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ─────────────────────────────────────────────────────────────
NB_TAG = "AddCalib_05b"

# ── Input: light curves with MJD (output of notebook 02) ─────────────────────
DIR_LC_IN = "./data_MergeVisits_02_out"
DIR_LC_PER_STAR_IN = os.path.join(DIR_LC_IN, "per_star")
GLOBAL_LC_FILE = "all_stars_lightcurves_mjd.csv"

# ── Output ────────────────────────────────────────────────────────────────────
DIR_DATA = f"./data_{NB_TAG}_out"
DIR_DATA_PER_STAR = os.path.join(DIR_DATA, "per_star")
DIR_FIGS = f"./figs_{NB_TAG}"

for d in [DIR_DATA, DIR_DATA_PER_STAR, DIR_FIGS]:
    os.makedirs(d, exist_ok=True)
    log.info("Directory ready: %s", d)

# ── Butler configuration (same as notebooks 01 / 98) ─────────────────────────
repo = "dp2_prep"
collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

# ── Timespan for Butler queries (wide window covering all DP2) ────────────────
# We keep a wide window; the exact visit is pinned by the visit.id constraint.
DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-01T00:00:00"
MJD_START = Time(DATE_START, format="isot", scale="utc").mjd
MJD_STOP = Time(DATE_STOP, format="isot", scale="utc").mjd
log.info("MJD window: [%.2f, %.2f]", MJD_START, MJD_STOP)

# ── Candidate dataset type names for the single-visit processed image ─────────
PVI_TABLE_CANDIDATES = [
    "preliminary_visit_image",
    "legacy_visit_image",
    "visit_image",
]

log.info("Configuration done.")

## 4. Initialise the Butler and select the PVI dataset type

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
log.info("Butler initialised | repo: %s", repo)

In [ ]:
# List visit-image-related dataset types for information
all_pvi_types = [
    d.name for d in registry.queryDatasetTypes() if "visit" in d.name.lower() and "image" in d.name.lower()
]
print("visit-image-related dataset types in registry:")
for t in sorted(all_pvi_types):
    print(f"  {t}")

In [ ]:
# Select the first registered PVI dataset type
PVI_DATASET = None
for name in PVI_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        PVI_DATASET = name
        log.info("Selected PVI dataset type: '%s'", PVI_DATASET)
        break

if PVI_DATASET is None:
    raise RuntimeError(
        "No recognised PVI dataset type found in this Butler collection. "
        f"Candidates tried: {PVI_TABLE_CANDIDATES}"
    )

## 5. Load the global merged light-curve table

In [ ]:
lc_path = os.path.join(DIR_LC_IN, GLOBAL_LC_FILE)
log.info("Loading: %s", lc_path)
df_all = pd.read_csv(lc_path)
log.info("Shape: %s  |  columns: %s", df_all.shape, df_all.columns.tolist())
df_all.head(3)

In [ ]:
df_all.groupby(["simbad_id", "band", "visit"])[["src_ra", "src_dec", "x", "y"]].mean()

In [ ]:
df_all.groupby(["simbad_id", "band", "visit"])[["src_ra", "src_dec", "x", "y"]].first()

In [ ]:
# Verify required columns are present.
# Note: 'detector' is NOT required here — it is unreliable (-1) and will be
# replaced by the value recovered from the Butler.
REQUIRED_COLS = ["visit", "src_ra", "src_dec", "x", "y", "band"]
missing = [c for c in REQUIRED_COLS if c not in df_all.columns]
if missing:
    raise ValueError(f"Required columns missing from input LC file: {missing}")
log.info("All required columns present: %s", REQUIRED_COLS)

# Ensure integer dtype for the Butler visit key
df_all["visit"] = df_all["visit"].astype(np.int64)

## 6. Build the Timespan for Butler spatial queries

In [ ]:
t1 = Time(MJD_START, format="mjd", scale="tai")
t2 = Time(MJD_STOP, format="mjd", scale="tai")
timespan = Timespan(t1, t2)
log.info("Timespan for Butler queries: MJD [%.1f, %.1f]  (delta=%.0f days)", t1.mjd, t2.mjd, t2.mjd - t1.mjd)

## 7. Helper: fetch the PVI for one (band, visit) using spatial overlap on (ra, dec)

The query uses:
- `visit.id = :visit`  — pins the exact visit
- `band.name = :band`  — selects the filter
- `visit_detector_region.region OVERLAPS POINT(:ra, :dec)` — resolves the
  correct detector without knowing its number in advance

For a well-formed dataset there is exactly **one** matching reference.
The function returns the loaded PVI and the extracted calibration dict,
or a failure dict if anything goes wrong.

In [ ]:
def fetch_calib_for_visit(
    butler: Butler,
    pvi_dataset: str,
    band: str,
    visit: int,
    ra: float,
    dec: float,
    x: float,
    y: float,
    timespan: Timespan,
) -> dict:
    """Retrieve the PVI that covers (ra, dec) during visit, and extract PhotoCalib.

    The detector number is discovered automatically from the spatial query;
    it does NOT need to be supplied.

    Returns
    -------
    dict with keys:
        calib_mean, calib_err, zeropoint,
        visit_from_image, detector_id, detector_name,
        photocal  (PhotoCalib object, kept for per-row getLocalCalibration),
        calib_ok  (bool).
    """

    pos = Point2D(x, y)

    result = dict(
        localCalib=np.nan,
        calib_mean=np.nan,
        calib_err=np.nan,
        zeropoint=np.nan,
        visit_from_image=-1,
        detector_id=-1,
        detector_name="",
        photocal=None,
        calib_ok=False,
    )

    # ── Spatial + visit query — no detector needed ────────────────────────────
    try:
        refs = list(
            butler.query_datasets(
                pvi_dataset,
                where=(
                    "visit.id = :visit "
                    "AND band.name = :band "
                    "AND visit.timespan OVERLAPS :timespan "
                    "AND visit_detector_region.region OVERLAPS POINT(:ra, :dec)"
                ),
                bind={"visit": int(visit), "band": band, "timespan": timespan, "ra": ra, "dec": dec},
            )
        )
    except Exception as exc:
        log.warning(
            "  query_datasets failed (band=%s visit=%d ra=%.4f dec=%.4f): %s", band, visit, ra, dec, exc
        )
        return result

    if len(refs) == 0:
        log.warning("  No PVI found  (band=%s visit=%d ra=%.4f dec=%.4f)", band, visit, ra, dec)
        return result

    if len(refs) > 1:
        log.warning(
            "  %d PVI refs found (expected 1) — using first  " "(band=%s visit=%d ra=%.4f dec=%.4f)",
            len(refs),
            band,
            visit,
            ra,
            dec,
        )

    # ── Load the image ────────────────────────────────────────────────────────
    try:
        pvi = butler.get(refs[0])
    except Exception as exc:
        log.warning("  butler.get failed (band=%s visit=%d): %s", band, visit, exc)
        return result

    # ── Extract PhotoCalib and Detector info ──────────────────────────────────
    try:
        photocal = pvi.getPhotoCalib()
        det_obj = pvi.getDetector()

        result["localCalib"] = photocal.getLocalCalibration(pos)
        result["photocal"] = photocal
        result["calib_mean"] = photocal.getCalibrationMean()
        result["calib_err"] = photocal.getCalibrationErr()
        result["zeropoint"] = 2.5 * np.log10(photocal.getInstFluxAtZeroMagnitude())
        result["visit_from_image"] = int(pvi.visitInfo.id)
        result["detector_id"] = int(det_obj.getId())
        result["detector_name"] = str(det_obj.getName())
        result["calib_ok"] = True

        log.debug(
            "  OK  band=%-3s visit=%-10d  det=%3d (%s)  " "local_calib=%.6g calib_mean=%.6g  zp=%.4f",
            band,
            visit,
            result["detector_id"],
            result["detector_name"],
            result["localCalib"],
            result["calib_mean"],
            result["zeropoint"],
        )
    except Exception as exc:
        log.warning("  PhotoCalib extraction failed (band=%s visit=%d): %s", band, visit, exc)

    return result

## 8. Fetch calibration for all unique (band, visit) pairs and cache results

For the spatial query we use the `src_ra` / `src_dec` of the first row
belonging to each `(band, visit)` group.  All sources sharing the same visit
and band necessarily fall on the same detector, so one query per pair suffices
for the CCD-level scalars (`calib_mean`, `calib_err`, `zeropoint`, detector
info).  The **local** calibration factor is computed row-by-row in section 10
using the cached `PhotoCalib` object.

In [ ]:
# One representative (ra, dec) per (band, visit) — first row of each group
representative = (
    df_all.groupby(["simbad_id", "band", "visit"])[["src_ra", "src_dec", "x", "y"]].first().reset_index()
)
log.info("Unique (band, visit) pairs to query: %d", len(representative))
representative.head()

In [ ]:
# Cache: key = (band, visit) → calib dict
calib_cache: dict = {}

n_ok = 0
n_err = 0

for idx, row in representative.iterrows():
    name = row["simbad_id"]
    band = row["band"]
    visit = int(row["visit"])
    ra = float(row["src_ra"])
    dec = float(row["src_dec"])
    x = float(row["x"])
    y = float(row["y"])
    key = (band, visit)

    if idx % 100 == 0:
        log.info(
            "[%d/%d] name= %s band=%-3s  visit=%-10d  (ra=%.4f, dec=%.4f)",
            idx + 1,
            len(representative),
            name,
            band,
            visit,
            ra,
            dec,
        )

    cdict = fetch_calib_for_visit(
        butler,
        PVI_DATASET,
        band=band,
        visit=visit,
        ra=ra,
        dec=dec,
        x=x,
        y=y,
        timespan=timespan,
    )
    calib_cache[key] = cdict

    if cdict["calib_ok"]:
        n_ok += 1
    else:
        n_err += 1

    gc.collect()

log.info("Calib fetch done: %d OK, %d errors out of %d pairs.", n_ok, n_err, len(representative))

## 9. Sanity check: visit number consistency

Verify that the visit id embedded in each PVI (`visitInfo.id`) matches
the visit number used in the query. Any mismatch signals a metadata issue.

In [ ]:
mismatches = []
for (band, visit), cdict in calib_cache.items():
    if not cdict["calib_ok"]:
        continue
    if cdict["visit_from_image"] != visit:
        mismatches.append(
            dict(
                band=band,
                visit_requested=visit,
                visit_in_image=cdict["visit_from_image"],
                detector_id=cdict["detector_id"],
            )
        )
        log.warning(
            "VISIT MISMATCH: requested=%d  image.visitInfo.id=%d  " "band=%s  detector=%d",
            visit,
            cdict["visit_from_image"],
            band,
            cdict["detector_id"],
        )

if mismatches:
    df_mismatches = pd.DataFrame(mismatches)
    print(f"WARNING — {len(mismatches)} visit mismatches found:")
    display(df_mismatches)
else:
    log.info("All visit numbers consistent between query and visitInfo.id — OK.")

## 10. Helper: attach calibration columns to a light-curve DataFrame

CCD-level scalars (`calib_mean`, `calib_err`, `zeropoint`, detector info) are
looked up from the cache via `(band, visit)`.  The **local** calibration at
the source pixel position `(x, y)` is computed row-by-row from the cached
`PhotoCalib` object — no extra Butler call is needed.

In [ ]:
def attach_calib_columns(df: pd.DataFrame, calib_cache: dict) -> pd.DataFrame:
    """Enrich a light-curve DataFrame with PhotoCalib and detector columns.

    Parameters
    ----------
    df          : LC DataFrame containing band, visit, x, y.
    calib_cache : dict mapping (band, visit) to calib dicts.

    Returns
    -------
    Copy of df with additional columns:
        calib_mean, calib_err, calib_local, zeropoint,
        visit_from_image, detector_id, detector_name.
    """
    df = df.copy()
    df["visit"] = df["visit"].astype(np.int64)

    # Pre-allocate new columns
    df["calib_mean"] = np.nan
    df["calib_err"] = np.nan
    df["calib_local"] = np.nan
    df["zeropoint"] = np.nan
    df["visit_from_image"] = -1
    df["detector_id"] = -1
    df["detector_name"] = ""

    for i, row in df.iterrows():
        key = (row["band"], int(row["visit"]))
        cdict = calib_cache.get(key)

        if cdict is None or not cdict["calib_ok"]:
            continue

        # CCD-level scalars (same for every source on this visit/detector)
        df.at[i, "calib_mean"] = cdict["calib_mean"]
        df.at[i, "calib_err"] = cdict["calib_err"]
        df.at[i, "zeropoint"] = cdict["zeropoint"]
        df.at[i, "visit_from_image"] = cdict["visit_from_image"]
        df.at[i, "detector_id"] = cdict["detector_id"]
        df.at[i, "detector_name"] = cdict["detector_name"]

        # Local calibration at the source pixel position (source-specific)
        try:
            pos = Point2D(float(row["x"]), float(row["y"]))
            df.at[i, "calib_local"] = cdict["photocal"].getLocalCalibration(pos)
        except Exception as exc:
            log.debug("getLocalCalibration failed at row %d: %s", i, exc)

    return df

## 11. Enrich and save the global light-curve table

In [ ]:
log.info("Attaching calibration columns to global LC table (%d rows)...", len(df_all))
df_all_calib = attach_calib_columns(df_all, calib_cache)
new_cols = [c for c in df_all_calib.columns if c not in df_all.columns]
log.info("New columns added: %s", new_cols)
df_all_calib[
    [
        "simbad_id",
        "visit",
        "band",
        "detector_id",
        "detector_name",
        "calib_mean",
        "calib_err",
        "calib_local",
        "zeropoint",
        "visit_from_image",
    ]
].head(5)

In [ ]:
def save_calib(df: pd.DataFrame, out_dir: str, basename: str) -> None:
    """Save enriched DataFrame as both CSV and Parquet."""
    csv_path = os.path.join(out_dir, basename + ".csv")
    parquet_path = os.path.join(out_dir, basename + ".parquet")
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)
    log.info("Saved CSV    : %s  (%d rows)", csv_path, len(df))
    log.info("Saved Parquet: %s", parquet_path)

In [ ]:
save_calib(df_all_calib, DIR_DATA, "all_stars_lightcurves_calib")

## 12. Process per-star files

In [ ]:
per_star_files = sorted(f for f in os.listdir(DIR_LC_PER_STAR_IN) if f.endswith("_lc_mjd.csv"))
log.info("Found %d per-star CSV files in %s", len(per_star_files), DIR_LC_PER_STAR_IN)
per_star_files[:5]

In [ ]:
n_ok_ps = 0
n_err_ps = 0

for fname in per_star_files:
    src_path = os.path.join(DIR_LC_PER_STAR_IN, fname)
    try:
        df_star = pd.read_csv(src_path)
        df_star["visit"] = df_star["visit"].astype(np.int64)
    except Exception as exc:
        log.error("ERROR reading %s: %s", fname, exc)
        n_err_ps += 1
        continue

    df_star_calib = attach_calib_columns(df_star, calib_cache)

    stem = fname.replace("_lc_mjd.csv", "")
    out_stem = stem + "_lc_calib"
    save_calib(df_star_calib, DIR_DATA_PER_STAR, out_stem)

    n_ok_ps += 1
    gc.collect()

log.info("Per-star calib done: %d OK, %d errors.", n_ok_ps, n_err_ps)

## 13. Diagnostics: calibration statistics per band

In [ ]:
calib_summary = (
    df_all_calib.dropna(subset=["calib_mean"])
    .groupby("band")
    .agg(
        n_rows=("calib_mean", "count"),
        calib_mean_med=("calib_mean", "median"),
        calib_mean_std=("calib_mean", "std"),
        calib_local_med=("calib_local", "median"),
        zeropoint_med=("zeropoint", "median"),
    )
)
print("Calibration summary per band:")
display(calib_summary)

In [ ]:
# Count of rows where the Butler call failed
n_missing = df_all_calib["calib_mean"].isna().sum()
n_total = len(df_all_calib)
log.info(
    "Rows without calib (Butler call failed): %d / %d  (%.1f %%)",
    n_missing,
    n_total,
    100.0 * n_missing / n_total if n_total else 0,
)

# Detector numbers actually found
print("\nDetector ids found per band:")
display(
    df_all_calib.dropna(subset=["calib_mean"])
    .groupby("band")["detector_id"]
    .value_counts()
    .rename("n_rows")
    .reset_index()
    .sort_values(["band", "detector_id"])
)

## 14. Quick-look plots

In [ ]:
def savefig(fig, name, dpi=150):
    """Save figure as both PDF and PNG."""
    fig.savefig(os.path.join(DIR_FIGS, name + ".pdf"), dpi=dpi, bbox_inches="tight")
    fig.savefig(os.path.join(DIR_FIGS, name + ".png"), dpi=dpi, bbox_inches="tight")
    log.info("Figure saved: %s (.pdf/.png)", os.path.join(DIR_FIGS, name))

In [ ]:
# ── Plot 1: distribution of calib_mean per band ───────────────────────────────
bands_available = sorted(df_all_calib["band"].dropna().unique())
n_bands = len(bands_available)
cmap = plt.get_cmap("tab10")

if n_bands > 0:
    fig, axes = plt.subplots(1, n_bands, figsize=(4 * n_bands, 4), sharey=False)
    if n_bands == 1:
        axes = [axes]
    for ax, band in zip(axes, bands_available):
        vals = df_all_calib.loc[df_all_calib["band"] == band, "calib_mean"].dropna()
        color = cmap(bands_available.index(band))
        ax.hist(vals, bins=40, color=color, alpha=0.75, edgecolor="white")
        ax.set_title(f"band = {band}")
        ax.set_xlabel("calib_mean  [nJy / ADU]")
        ax.set_ylabel("count")
    fig.suptitle("Distribution of PhotoCalib mean per band", fontsize=12)
    plt.tight_layout()
    savefig(fig, "calib_mean_distribution_per_band")
    plt.show()

In [ ]:
# ── Plot 2: zero-point vs expMidptMJD per band ────────────────────────────────
if "expMidptMJD" in df_all_calib.columns and n_bands > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    for band in bands_available:
        sub = (
            df_all_calib[(df_all_calib["band"] == band) & df_all_calib["zeropoint"].notna()].drop_duplicates(
                subset=["visit"]
            )  # one point per visit
        )
        color = cmap(bands_available.index(band))
        ax.scatter(sub["expMidptMJD"], sub["zeropoint"], s=10, alpha=0.6, label=band, color=color)
    ax.set_xlabel("expMidptMJD")
    ax.set_ylabel("Zero-point  [AB mag]")
    ax.set_title("Photometric zero-point vs. time — one point per visit")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(fig, "zeropoint_vs_mjd")
    plt.show()

In [ ]:
# ── Plot 3: calib_mean vs expMidptMJD per band ────────────────────────────────
if "expMidptMJD" in df_all_calib.columns and n_bands > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    for band in bands_available:
        sub = df_all_calib[
            (df_all_calib["band"] == band) & df_all_calib["calib_mean"].notna()
        ].drop_duplicates(subset=["visit"])
        color = cmap(bands_available.index(band))
        ax.scatter(sub["expMidptMJD"], sub["calib_mean"], s=10, alpha=0.6, label=band, color=color)
    ax.set_xlabel("expMidptMJD")
    ax.set_ylabel("calib_mean  [nJy / ADU]")
    ax.set_title("PhotoCalib mean vs. time — one point per visit")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(fig, "calib_mean_vs_mjd")
    plt.show()

In [ ]:
# ── Plot 4: calib_local vs calib_mean (spatial variation across the CCD) ──────
if n_bands > 0:
    fig, ax = plt.subplots(figsize=(6, 6))
    for band in bands_available:
        sub = df_all_calib[
            (df_all_calib["band"] == band)
            & df_all_calib["calib_mean"].notna()
            & df_all_calib["calib_local"].notna()
        ]
        color = cmap(bands_available.index(band))
        ax.scatter(sub["calib_mean"], sub["calib_local"], s=5, alpha=0.4, label=band, color=color)
    # 1:1 reference line
    lims_min = df_all_calib[["calib_mean", "calib_local"]].min().min()
    lims_max = df_all_calib[["calib_mean", "calib_local"]].max().max()
    ax.plot([lims_min, lims_max], [lims_min, lims_max], "k--", lw=1, label="1:1")
    ax.set_xlabel("calib_mean  [nJy / ADU]")
    ax.set_ylabel("calib_local  [nJy / ADU]")
    ax.set_title("Local vs. mean PhotoCalib factor")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(fig, "calib_local_vs_mean")
    plt.show()

## 15. Output directory listing

In [ ]:
print(f"\n=== Contents of {DIR_DATA} ===")
for entry in sorted(os.listdir(DIR_DATA)):
    full = os.path.join(DIR_DATA, entry)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f"  [DIR]  {entry}/  ({n} files)")
    else:
        size_kb = os.path.getsize(full) / 1024
        print(f"  [FILE] {entry}  ({size_kb:.1f} kB)")